# Assignment 18: L1 and L2 Regularization (Melbourne Housing)

**Objective**: Apply L1 (Lasso) and L2 (Ridge) Regularization to the Melbourne Housing Market dataset to reduce overfitting and improve model generalization.

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

# Set display options
pd.set_option('display.max_columns', None)

## 1. Load Data

In [16]:
df = pd.read_csv('Melbourne_housing_FULL.csv')
print(f"Shape: {df.shape}")
df.head()

Shape: (34857, 21)


,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,68 Studley St,2,h,NaN,SS,Jellis,3/09/2016,2.5,3067.0,2.0,1.0,1.0,126.0,NaN,NaN,Yarra City Council,-37.8014,144.9958,Northern Metropolitan,4019.0
1,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,3/12/2016,2.5,3067.0,2.0,1.0,1.0,202.0,NaN,NaN,Yarra City Council,-37.7996,144.9984,Northern Metropolitan,4019.0
2,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,4/02/2016,2.5,3067.0,2.0,1.0,0.0,156.0,79.0,1900.0,Yarra City Council,-37.8079,144.9934,Northern Metropolitan,4019.0
3,Abbotsford,18/659 Victoria St,3,u,NaN,VB,Rounds,4/02/2016,2.5,3067.0,3.0,2.0,1.0,0.0,NaN,NaN,Yarra City Council,-37.8114,145.0116,Northern Metropolitan,4019.0
4,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,4/03/2017,2.5,3067.0,3.0,2.0,0.0,134.0,150.0,1900.0,Yarra City Council,-37.8093,144.9944,Northern Metropolitan,4019.0


## 2. Preprocessing
- Drop rows where the target 'Price' is missing.
- Define Features (X) and Target (y).
- Split into Train and Test sets.
- Create a Preprocessing Pipeline for Numeric (Impute + Scale) and Categorical (Impute + OneHot) features.

In [17]:
# Drop rows with missing Price
df.dropna(subset=['Price'], inplace=True)

# Define X and y
X = df.drop('Price', axis=1)
y = df['Price']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

Train shape: (21797, 20)
Test shape: (5450, 20)


In [18]:
# Identify Numeric and Categorical columns
cols_to_use = ['Suburb', 'Rooms', 'Type', 'Method', 'SellerG', 'Regionname', 'Propertycount', 'Distance', 'CouncilArea', 'Bedroom2', 'Bathroom', 'Car', 'Landsize', 'BuildingArea']
numeric_features = [col for col in X.select_dtypes(include=['int64', 'float64']).columns if col in cols_to_use]
categorical_features = [col for col in X.select_dtypes(include=['object']).columns if col in cols_to_use]

print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")

# Create Transformers
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine into Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='drop'    
)

Numeric features: 8
Categorical features: 6


## 3. Model Implementation
We will implement three models:
1. **Linear Regression**: The baseline model.
2. **Lasso Regression (L1)**: Adds absolute value of magnitude of coefficient as penalty term to the loss function.
3. **Ridge Regression (L2)**: Adds squared magnitude of coefficient as penalty term to the loss function.

In [19]:
models = {
    'Linear Regression': LinearRegression(),
    'Lasso (L1)': Lasso(alpha=50, max_iter=100, tol=0.01), # Alpha controls regularization strength
    'Ridge (L2)': Ridge(alpha=50, max_iter=100, tol=0.01),
    'RandomForestRegressor': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
}

results = []

for name, model in models.items():
    # Create full pipeline
    clf = Pipeline(steps=[('preprocessor', preprocessor),
                          ('regressor', model)])
    
    # Fit model
    clf.fit(X_train, y_train)
    
    # Predict
    y_train_pred = clf.predict(X_train)
    y_test_pred = clf.predict(X_test)
    
    # Evaluate
    train_score = r2_score(y_train, y_train_pred)
    test_score = r2_score(y_test, y_test_pred)
    
    results.append({
        'Model': name,
        'Train R2': train_score,
        'Test R2': test_score,
    })
    
    print(f"{name} - Train R2: {train_score:.4f}, Test R2: {test_score:.4f}")

Linear Regression - Train R2: 0.6858, Test R2: 0.6489


/opt/homebrew/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.786e+14, tolerance: 8.882e+13
  model = cd_fast.enet_coordinate_descent(


Lasso (L1) - Train R2: 0.6814, Test R2: 0.6511
Ridge (L2) - Train R2: 0.6708, Test R2: 0.6390
RandomForestRegressor - Train R2: 0.8454, Test R2: 0.7547


## 4. Comparison and Conclusion

In [21]:
results_df = pd.DataFrame(results)
results_df

,Model,Train R2,Test R2
0,Linear Regression,0.685764,0.648900
1,Lasso (L1),0.681417,0.651146
2,Ridge (L2),0.670843,0.638961
3,RandomForestRegressor,0.845361,0.754656


### Analysis
- **Linear Regression** often overfits if there are many features or multicollinearity.
- **Lasso (L1)** can shrink coefficients to zero, effectively performing feature selection.
- **Ridge (L2)** shrinks coefficients towards zero but rarely to exactly zero, handling multicollinearity well.
- **RandomForestRegressor** is less sensitive to feature scaling and handles multicollinearity well.
By adjusting `alpha`, we can control the strength of regularization. Higher alpha = more regularization.